### **Hyperparameter Tuning the ANN using Optuna**

In [71]:
%pip install optuna

In [72]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [73]:
torch.manual_seed(42)

In [74]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device is {device}")

Device is cuda


In [75]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [76]:
dataset_path = "/content/drive/MyDrive/DataSets/fashion-mnist_train.csv"

In [77]:
df = pd.read_csv(dataset_path)

In [78]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [79]:
X = df.drop(columns="label")
y = df["label"]

In [80]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [81]:
# scaling 
X_train = X_train / 255.0
X_test = X_test / 255.0

In [82]:
len(X_train)

48000

In [83]:
# CustomDatset class
class CustomDatset(Dataset):
    def __init__(self,features, labels):
        self.features = torch.tensor(features.to_numpy(),dtype=torch.float32)
        self.labels = torch.tensor(labels.to_numpy(),dtype=torch.long)
    def __len__(self):
        return len(self.features)
    def __getitem__(self,idx):
        return self.features[idx], self.labels[idx]

In [84]:
# creating training dataset and testing dataset object from CustomDatset class
training_dataset = CustomDatset(X_train,y_train)
testing_datset = CustomDatset(X_test,y_test)

In [85]:
# creating test loader and train loader object from DataLoader Class
train_loader = DataLoader(training_dataset,batch_size=32,shuffle=True, pin_memory=True)
test_loader = DataLoader(testing_datset,batch_size=32,shuffle=False,pin_memory=False)

In [86]:
for i in train_loader:
    print(i)
    break

[tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]]), tensor([6, 2, 7, 5, 2, 3, 3, 9, 4, 1, 5, 4, 7, 7, 7, 9, 6, 3, 7, 7, 2, 4, 8, 5,
        9, 1, 6, 1, 4, 0, 3, 3])]


In [87]:
for i in test_loader:
    print(i)
    break

[tensor([[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.8275, 0.4000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]]), tensor([7, 8, 8, 5, 9, 1, 2, 6, 6, 2, 5, 0, 7, 1, 6, 0, 6, 2, 9, 1, 2, 4, 8, 0,
        4, 9, 1, 0, 0, 5, 1, 6])]


In [88]:
# creating ann class
class MyANN(nn.Module):
    def __init__(self,num_features):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features,128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(128,64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(64,32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(32,10)
        )
    
    def forward(self,features):
        return self.model(features)

In [89]:
# learning rate and epochs
learning_rate = 0.1
epochs = 50

In [90]:
# instantiate model and move to gpu
model = MyANN(X_train.shape[1])
model = model.to(device)

# loss function
criterion = nn.CrossEntropyLoss()

# optimizer
optimizer = optim.SGD(model.parameters(),lr=learning_rate,weight_decay= 1e-5)

In [91]:
# training loop
for epoch in range(epochs):
    total_epoch_loss = 0
    for batch_features, batch_labels in train_loader:
        # move data to gpu
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

        # forward pass
        output = model(batch_features)

        # calculating loss
        loss = criterion(output,batch_labels)

        # back pass
        optimizer.zero_grad()
        loss.backward()

        # gradients update
        optimizer.step()

        # total epoch loss
        total_epoch_loss = total_epoch_loss + loss.item()

    # avg loss
    avg_loss = total_epoch_loss / len(train_loader)
    print(f"Epochs:{epoch+1} | Loss:{avg_loss:.4f}")

        

Epochs:1 | Loss:0.7629
Epochs:2 | Loss:0.5894
Epochs:3 | Loss:0.5441
Epochs:4 | Loss:0.5154
Epochs:5 | Loss:0.5001
Epochs:6 | Loss:0.4819
Epochs:7 | Loss:0.4712
Epochs:8 | Loss:0.4563
Epochs:9 | Loss:0.4470
Epochs:10 | Loss:0.4398
Epochs:11 | Loss:0.4326
Epochs:12 | Loss:0.4244
Epochs:13 | Loss:0.4158
Epochs:14 | Loss:0.4080
Epochs:15 | Loss:0.4153
Epochs:16 | Loss:0.4078
Epochs:17 | Loss:0.4024
Epochs:18 | Loss:0.3977
Epochs:19 | Loss:0.3932
Epochs:20 | Loss:0.3955
Epochs:21 | Loss:0.3885
Epochs:22 | Loss:0.3857
Epochs:23 | Loss:0.3839
Epochs:24 | Loss:0.3781
Epochs:25 | Loss:0.3797
Epochs:26 | Loss:0.3724
Epochs:27 | Loss:0.3713
Epochs:28 | Loss:0.3709
Epochs:29 | Loss:0.3689
Epochs:30 | Loss:0.3663
Epochs:31 | Loss:0.3621
Epochs:32 | Loss:0.3583
Epochs:33 | Loss:0.3579
Epochs:34 | Loss:0.3579
Epochs:35 | Loss:0.3533
Epochs:36 | Loss:0.3558
Epochs:37 | Loss:0.3463
Epochs:38 | Loss:0.3457
Epochs:39 | Loss:0.3477
Epochs:40 | Loss:0.3506
Epochs:41 | Loss:0.3488
Epochs:42 | Loss:0.3455
E

In [92]:
model.eval()

MyANN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.3, inplace=False)
    (12): Linear(in_features=32, out_features=10, bias=True)
  )
)

In [93]:
# evaluation on test data
total = 0
correct = 0

with torch.no_grad():
    for batch_features,batch_labels in test_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        outputs = model(batch_features)
        _, predicted = torch.max(outputs,1)

        total = total + batch_labels.shape[0]
        correct = correct + (predicted == batch_labels).sum().item()
    print(correct/total)


0.8916666666666667


In [94]:
# simple evaluation
correct = 0
total = 0

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        y = y.to(device)

        pred = model(X).argmax(dim=1)

        correct += (pred == y).sum().item()
        total += len(y)

accuracy = correct / total
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.8917


In [95]:
# simple evaluation on train data
correct = 0
total = 0

with torch.no_grad():
    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device)

        pred = model(X).argmax(dim=1)

        correct += (pred == y).sum().item()
        total += len(y)

accuracy = correct / total
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.9304
